# Classification model

In [1]:
import psycopg2
import datetime as dt
import numpy as np
import pandas as pd
import logging
import json

In [2]:
# Configure logging
logging.basicConfig(filename='model_training.log', level=logging.DEBUG, 
                    format='%(asctime)s %(levelname)s:%(message)s')

In [3]:
# Load database configuration from JSON
with open('../db_config.json') as config_file:
    db_params = json.load(config_file)
 
# Connect to the database
try:
    conn = psycopg2.connect(**db_params)
    cursor = conn.cursor()
    logging.info('Connection to database established')
except OperationalError as e:
    logging.error('Connection to database could not be established.')

INFO: Connection to database established


In [4]:
get_data = '''SELECT * FROM group14_warehouse.regression_data'''

cursor.execute(get_data)
logging.info('Data pulled from warehouse')

INFO: Data pulled from warehouse


In [5]:
df = pd.DataFrame(cursor.fetchall(), columns=['datetime_s', 'datetime_e', 'event_cat', 'event_sev', 'speed', 'end_speed',
                                                   'maxwaarde', 'streetname', 'rain_intensity', 'temperature', 'windspeed',
                                                   'speed_limit', 'light_condition', 'accident_sev', 'accident_prob'])

print(df.head())
conn.close()
logging.info('Connection to database closed')

INFO: Connection to database closed


               datetime_s              datetime_e        event_cat event_sev  \
0 2018-01-01 00:21:41.100 2018-01-01 00:21:43.500  HARSH CORNERING       HC1   
1 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500            SPEED       SP1   
2 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500            SPEED       SP1   
3 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500            SPEED       SP1   
4 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500            SPEED       SP1   

       speed  end_speed  maxwaarde         streetname  rain_intensity  \
0  30.577536  37.014910   0.854562       Doornboslaan             0.0   
1  82.076546  82.076546  86.904580  Backer en Ruebweg             0.0   
2  82.076546  82.076546  86.904580  Backer en Ruebweg             0.0   
3  82.076546  82.076546  86.904580  Backer en Ruebweg             0.0   
4  82.076546  82.076546  86.904580  Backer en Ruebweg             0.0   

   temperature  windspeed speed_limit light_condition          accident_sev  \
0

## Place for functions

In [6]:
# Function to split the data into training, validation and testing partitions
def split_tvt(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X,
                                                        y,
                                                        test_size = 1/10,
                                                        random_state = 0)

    X_train, X_val, y_train, y_val = train_test_split(X_train,
                                                      y_train,
                                                      test_size = 1/9,
                                                      random_state = 0)

    print("X_train shape: ", X_train.shape, " ||  y_train shape: ", y_train.shape)
    print("X_val shape: ", X_val.shape, " ||  y_val shape: ", y_val.shape)
    print("X_test shape: ", X_test.shape, " ||  y_test shape: ", y_test.shape)
    return X_train, X_val, X_test, y_train, y_val, y_test

In [7]:
# Function used to plot the loss over epochs for all models
def loss_plotter(H):
    plt.plot(range(1, len(H.history['loss']) + 1), H.history['loss'], label='train_loss')
    plt.plot(range(1, len(H.history['val_loss']) + 1), H.history['val_loss'], label='val_loss')
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('Epoch')
    plt.ylim(0, 4)
    plt.xticks(np.arange(1, len(H.history['loss'])+1))
    plt.legend()
    plt.show()

# Function used to plot the accuracy over epochs for all models
def accuracy_plotter(H):
    plt.plot(range(1, len(H.history['accuracy']) + 1), H.history['accuracy'], label='train_acc')
    plt.plot(range(1, len(H.history['val_accuracy']) + 1), H.history['val_accuracy'], label='val_acc')
    plt.title('model accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epochs')
    plt.xticks(np.arange(1, len(H.history['categorical_accuracy'])+1))
    plt.legend()
    plt.show()

In [8]:
def encode_categoricals(categorical_columns):
    for column in categorical_columns: # Loop through categorical columns
        encoder = LabelEncoder()
        df[column] = encoder.fit_transform(df[column]) # Encode categories using Labelencoder. 

### Preprocessing

In [9]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

import datetime as dt

import matplotlib.pyplot as plt

In [10]:
# Check for missing values (none expected, but checking is a good habit)
print(F"Missing:\n{df.isna().sum()}")

# Outliers not removed as there is already too many, and we need them for part of the prediction.  
# However some of the outliers have been removed from the rain intensity column in the creation of the warehouse table. 
# This was done by cross validating both the electric and normal rain meter.

Missing:
datetime_s              0
datetime_e              0
event_cat               0
event_sev               0
speed                   0
end_speed               0
maxwaarde               0
streetname              0
rain_intensity          0
temperature             0
windspeed               0
speed_limit        930527
light_condition    930527
accident_sev        17086
accident_prob       17086
dtype: int64


In [11]:
df['datetime_s'] = pd.to_datetime(df['datetime_s']) # Make sure datetime column is actual datetime format

df['datetime_e'] = pd.to_datetime(df['datetime_e']) # Make sure datetime column is actual datetime format

cats = ['event_cat', 'event_sev', 'streetname']     # Define categorical columns
encode_categoricals(cats)

print(df.head())

               datetime_s              datetime_e  event_cat  event_sev  \
0 2018-01-01 00:21:41.100 2018-01-01 00:21:43.500          2          2   
1 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500          3          4   
2 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500          3          4   
3 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500          3          4   
4 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500          3          4   

       speed  end_speed  maxwaarde  streetname  rain_intensity  temperature  \
0  30.577536  37.014910   0.854562         303             0.0          8.8   
1  82.076546  82.076546  86.904580          72             0.0          8.8   
2  82.076546  82.076546  86.904580          72             0.0          8.8   
3  82.076546  82.076546  86.904580          72             0.0          8.8   
4  82.076546  82.076546  86.904580          72             0.0          8.8   

   windspeed speed_limit light_condition          accident_sev  accident_p

### Feature engineering

I need a List containing holidays from 2018 to 2024 to see if given date is a holiday, as this might influence behaviour on the road.

In [12]:
holidays_nl = [
    # 2018
    ["2018-01-01", "New Year's Day"],
    ["2018-03-30", "Good Friday"],
    ["2018-04-01", "Easter Sunday"],
    ["2018-04-02", "Easter Monday"],
    ["2018-04-27", "King's Birthday"],
    ["2018-05-04", "National Remembrance Day"],
    ["2018-05-05", "Liberation Day"],
    ["2018-05-10", "Ascension Day"],
    ["2018-05-20", "Whit Sunday"],
    ["2018-05-21", "Whit Monday"],
    ["2018-12-25", "Christmas Day"],
    ["2018-12-26", "St. Stephen's Day"],

    # 2019
    ["2019-01-01", "New Year's Day"],
    ["2019-04-19", "Good Friday"],
    ["2019-04-21", "Easter Sunday"],
    ["2019-04-22", "Easter Monday"],
    ["2019-04-27", "King's Birthday"],
    ["2019-05-04", "National Remembrance Day"],
    ["2019-05-05", "Liberation Day"],
    ["2019-05-30", "Ascension Day"],
    ["2019-06-09", "Whit Sunday"],
    ["2019-06-10", "Whit Monday"],
    ["2019-12-25", "Christmas Day"],
    ["2019-12-26", "St. Stephen's Day"],

    # 2020
    ["2020-01-01", "New Year's Day"],
    ["2020-04-10", "Good Friday"],
    ["2020-04-12", "Easter Sunday"],
    ["2020-04-13", "Easter Monday"],
    ["2020-04-27", "King's Birthday"],
    ["2020-05-04", "National Remembrance Day"],
    ["2020-05-05", "Liberation Day"],
    ["2020-05-21", "Ascension Day"],
    ["2020-05-31", "Whit Sunday"],
    ["2020-06-01", "Whit Monday"],
    ["2020-12-25", "Christmas Day"],
    ["2020-12-26", "St. Stephen's Day"],

    # 2021
    ["2021-01-01", "New Year's Day"],
    ["2021-04-02", "Good Friday"],
    ["2021-04-04", "Easter Sunday"],
    ["2021-04-05", "Easter Monday"],
    ["2021-04-27", "King's Birthday"],
    ["2021-05-04", "National Remembrance Day"],
    ["2021-05-05", "Liberation Day"],
    ["2021-05-13", "Ascension Day"],
    ["2021-05-23", "Whit Sunday"],
    ["2021-05-24", "Whit Monday"],
    ["2021-12-25", "Christmas Day"],
    ["2021-12-26", "St. Stephen's Day"],

    # 2022
    ["2022-01-01", "New Year's Day"],
    ["2022-04-15", "Good Friday"],
    ["2022-04-17", "Easter Sunday"],
    ["2022-04-18", "Easter Monday"],
    ["2022-04-27", "King's Birthday"],
    ["2022-05-04", "National Remembrance Day"],
    ["2022-05-05", "Liberation Day"],
    ["2022-05-26", "Ascension Day"],
    ["2022-06-05", "Whit Sunday"],
    ["2022-06-06", "Whit Monday"],
    ["2022-12-25", "Christmas Day"],
    ["2022-12-26", "St. Stephen's Day"],

    # 2023
    ["2023-01-01", "New Year's Day"],
    ["2023-04-07", "Good Friday"],
    ["2023-04-09", "Easter Sunday"],
    ["2023-04-10", "Easter Monday"],
    ["2023-04-27", "King's Birthday"],
    ["2023-05-04", "National Remembrance Day"],
    ["2023-05-05", "Liberation Day"],
    ["2023-05-18", "Ascension Day"],
    ["2023-05-28", "Whit Sunday"],
    ["2023-05-29", "Whit Monday"],
    ["2023-12-25", "Christmas Day"],
    ["2023-12-26", "St. Stephen's Day"],

    # 2024
    ["2024-01-01", "New Year's Day"],
    ["2024-03-29", "Good Friday"],
    ["2024-03-31", "Easter Sunday"],
    ["2024-04-01", "Easter Monday"],
    ["2024-04-27", "King's Birthday"],
    ["2024-05-04", "National Remembrance Day"],
    ["2024-05-05", "Liberation Day"],
    ["2024-05-09", "Ascension Day"],
    ["2024-05-19", "Whit Sunday"],
    ["2024-05-20", "Whit Monday"],
    ["2024-12-25", "Christmas Day"],
    ["2024-12-26", "St. Stephen's Day"]
]

# List provided by blackbox.ai, confirmed on www.government.nl

In [13]:
df['is_holiday'] = df["datetime_s"].dt.date.isin([pd.to_datetime(holiday[0]).date() for holiday in holidays_nl])

print(df['is_holiday'].value_counts())

is_holiday
False    5772968
True      156620
Name: count, dtype: int64


In [14]:
df['weekday'] = df['datetime_s'].dt.dayofweek

print(df['weekday'].value_counts())

weekday
4    961550
3    909128
2    908721
1    882159
0    829217
5    799561
6    639252
Name: count, dtype: int64


In [15]:
df['maxspeed'] = np.nan
df['gforce'] = np.nan

df.loc[df['event_cat'] == 3, 'maxspeed'] = df['maxwaarde']
df.loc[df['event_cat'] != 3, 'gforce'] = df['maxwaarde']

print(df[['maxspeed', 'gforce']].head())

   maxspeed    gforce
0       NaN  0.854562
1  86.90458       NaN
2  86.90458       NaN
3  86.90458       NaN
4  86.90458       NaN


In [16]:
df['hour'] = df['datetime_s'].dt.hour

In [17]:
df['duration'] = df['datetime_e'] - df['datetime_s']

df['duration_in_s'] = df['duration'].dt.total_seconds()

In [18]:
# Calculate the change in speed
df['delta_v'] = df['maxspeed'] - df['speed']

# Calculate the acceleration
df['acceleration'] = df['delta_v'] / df['duration_in_s']

# Calculate the g-force
df['calculated_gforce'] = df['acceleration'] / 9.81

# Fill NaN values in the original gforce column with the calculated values
df['gforce'].fillna(df['calculated_gforce'], inplace=True)

# Drop the intermediate calculation columns if desired
df.drop(columns=['delta_v', 'acceleration', 'calculated_gforce'], inplace=True)

/tmp/ipykernel_530987/2473633758.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['gforce'].fillna(df['calculated_gforce'], inplace=True)


In [19]:
df = df.drop(['datetime_s', 'datetime_e', 'maxwaarde', 'duration', 'maxspeed'], axis=1)

In [20]:
print(df.columns)

Index(['event_cat', 'event_sev', 'speed', 'end_speed', 'streetname',
       'rain_intensity', 'temperature', 'windspeed', 'speed_limit',
       'light_condition', 'accident_sev', 'accident_prob', 'is_holiday',
       'weekday', 'gforce', 'hour', 'duration_in_s'],
      dtype='object')


In [24]:
def set_speed_limit(df):
    # Replace "Un" with "0" 
    df['speed_limit'] = df['speed_limit'].astype(str).str.replace("Un", "0", regex=False) 

    # Replace "No" with "0"
    df['speed_limit'] = df['speed_limit'].astype(str).str.replace("No", "0", regex=False)

    # Drop rows where speed_limit is still not numeric after the replacements
    # this will allow us to convert it back to an int
    df = df[pd.to_numeric(df['speed_limit'], errors='coerce').notnull()]


    # Convert speed_limit back to integer 
    df['speed_limit'] = df['speed_limit'].astype(int)
    
    return df

df = set_speed_limit(df)
print(df['speed_limit'])

0           0
1           0
2           0
3           0
4          70
           ..
5912487    50
5912493    50
5912496    70
5912498    50
5912500    70
Name: speed_limit, Length: 4999061, dtype: int64


/tmp/ipykernel_530987/3527637832.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['speed_limit'] = df['speed_limit'].astype(int)


In [27]:
X = df.drop(['accident_sev', 'accident_prob'], axis=1)
y = df['event_cat']
y_sev = df['event_sev']

print(X.head())

   event_cat  event_sev      speed  end_speed  streetname  rain_intensity  \
0          2          2  30.577536  37.014910         303             0.0   
1          3          4  82.076546  82.076546          72             0.0   
2          3          4  82.076546  82.076546          72             0.0   
3          3          4  82.076546  82.076546          72             0.0   
4          3          4  82.076546  82.076546          72             0.0   

   temperature  windspeed  speed_limit light_condition  is_holiday  weekday  \
0          8.8      12.87            0        Daylight        True        0   
1          8.8      12.87            0        Darkness        True        0   
2          8.8      12.87            0        Daylight        True        0   
3          8.8      12.87            0        Daylight        True        0   
4          8.8      12.87           70        Daylight        True        0   

     gforce  hour  duration_in_s  
0  0.854562     0          

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
# Model training
model = RandomForestClassifier(n_estimators=100, random_state=42)
logging.info('Model Training started: Random Forest Classifier')
model.fit(X_train, y_train)
logging.info('Model Training finished: Random Forest Classifier')

INFO: Model Training started: Random Forest Classifier
INFO: Model Training finished: Random Forest Classifier


In [24]:
# Model evaluation
logging.info('Model Evaluation: Random Forest Classifier')
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

INFO: Model Evaluation: Random Forest Classifier


              precision    recall  f1-score   support

           0       0.89      0.10      0.19       668
           1       0.91      0.81      0.86      4762
           2       0.98      1.00      0.99     66249
           3       1.00      1.00      1.00     64864

    accuracy                           0.99    136543
   macro avg       0.95      0.73      0.76    136543
weighted avg       0.99      0.99      0.99    136543



In [25]:
# Probability predictions
probabilities = model.predict_proba(X_test)
print(probabilities[0])

[0. 0. 0. 1.]


# Neural Network

## Additional

In [32]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense

2024-06-11 13:22:26.742400: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-11 13:22:26.742452: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-11 13:22:26.744288: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-06-11 13:22:26.751315: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [33]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

In [34]:
# Oversampling (SMOTE)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# # Undersampling
# rus = RandomUnderSampler(random_state=42)
# X_resampled, y_resampled = rus.fit_resample(X, y)

In [35]:
y_resampled = pd.get_dummies(y_resampled)

In [36]:
print(pd.DataFrame(y_resampled).value_counts())

0      1      2      3    
False  False  False  True     331592
              True   False    331592
       True   False  False    331592
True   False  False  False    331592
Name: count, dtype: int64


### Split data

In [37]:
X_v2_train, X_v2_val, X_v2_test, y_v2_train, y_v2_val, y_v2_test = split_tvt(X_resampled, y_resampled)

X_train shape:  (1061094, 9)  ||  y_train shape:  (1061094, 4)
X_val shape:  (132637, 9)  ||  y_val shape:  (132637, 4)
X_test shape:  (132637, 9)  ||  y_test shape:  (132637, 4)


In [38]:
X_v2_train = tf.convert_to_tensor(X_v2_train, dtype=tf.float32)
y_v2_train = tf.convert_to_tensor(y_v2_train, dtype=tf.float32)
X_v2_val = tf.convert_to_tensor(X_v2_val, dtype=tf.float32)
y_v2_val = tf.convert_to_tensor(y_v2_val, dtype=tf.float32)
X_v2_test = tf.convert_to_tensor(X_v2_test, dtype=tf.float32)
y_v2_test = tf.convert_to_tensor(y_v2_test, dtype=tf.float32)

2024-06-11 13:22:40.267034: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
2024-06-11 13:22:40.267449: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 44449 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:01:00.0, compute capability: 8.6


## Model Architecture and training

In [39]:
from tensorflow.python.client import device_lib 
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 9120841939318582106
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 46608678912
locality {
  bus_id: 1
  links {
  }
}
incarnation: 15327962268311133784
physical_device_desc: "device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:01:00.0, compute capability: 8.6"
xla_global_id: 416903419
]


2024-06-11 13:22:41.255330: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:0 with 44449 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:01:00.0, compute capability: 8.6


In [44]:
# Define the model
model = Sequential()

# Input layer with 9 neurons (for the 9 features)
model.add(Dense(16, input_dim=9, activation='relu')) 

# Hidden layer (adjust units as needed)
model.add(Dense(16, activation='relu')) 

# Output layer with 4 neurons (for 4 categories)
model.add(Dense(4, activation='softmax')) 

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [45]:
logging.info('Model training started: Basic 4-layer Neural Network')
H_nn = model.fit(X_v2_train, y_v2_train, epochs=15, batch_size=64, validation_data=(X_v2_val, y_v2_val))
logging.info('Model training finished: Basic 4-layer Neural Network')

INFO: Model training started: Basic 4-layer Neural Network


Epoch 1/15


2024-06-11 13:27:09.094210: I external/local_xla/xla/service/service.cc:168] XLA service 0x7efdf1d852c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-06-11 13:27:09.094267: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2024-06-11 13:27:09.103209: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-06-11 13:27:09.424965: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8906
I0000 00:00:1718112429.563987 2188049 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


16580/16580 [==============================] - 40s 2ms/step - loss: 0.7570 - accuracy: 0.6857 - val_loss: 0.5941 - val_accuracy: 0.7501
Epoch 2/15
16580/16580 [==============================] - 36s 2ms/step - loss: 0.5327 - accuracy: 0.7632 - val_loss: 0.4494 - val_accuracy: 0.7841
Epoch 3/15
16580/16580 [==============================] - 38s 2ms/step - loss: 0.4817 - accuracy: 0.7746 - val_loss: 0.5287 - val_accuracy: 0.7579
Epoch 4/15
16580/16580 [==============================] - 36s 2ms/step - loss: 0.4632 - accuracy: 0.7824 - val_loss: 0.4629 - val_accuracy: 0.7841
Epoch 5/15
16580/16580 [==============================] - 37s 2ms/step - loss: 0.4446 - accuracy: 0.7892 - val_loss: 0.4387 - val_accuracy: 0.7920
Epoch 6/15
16580/16580 [==============================] - 39s 2ms/step - loss: 0.4292 - accuracy: 0.7935 - val_loss: 0.4569 - val_accuracy: 0.7804
Epoch 7/15
16580/16580 [==============================] - 40s 2ms/step - loss: 0.4204 - accuracy: 0.7963 - val_loss: 0.4070 - val

INFO: Model training finished: Basic 4-layer Neural Network


In [ ]:
results = model.evaluate(X_v2_test, y_v2_test, batch_size=64)
logging.info(F'Model Evaluation: Basic 4-layer Neural Network \ntest loss: {str(results[0])}, test accuracy: {str(results[1])}')

accuracy_plotter(H_nn)
loss_plotter(H_nn)

In [ ]:
# Model evaluation
logging.info('Model Evaluation: Basic 4-layer Neural Network')
y_pred = pd.DataFrame(model.predict(X_v2_test)).idxmax(axis=1)
print(classification_report(pd.DataFrame(y_v2_test).idxmax(axis=1), y_pred))